# Model Training

The source supervised experiments use five drive-specific classifiers. The source’s Random Forest notebook uses an 80/20 chronological class-wise split; its XGBoost notebook uses 60/20/20 train/validation/test. The source resampling code samples failure rows from the complete positive set after the split, so it can include held-out positives. This leakage is retained and explicitly recorded here for reproduction, not recommended as a deployment practice.

## Imports and Paths

In [111]:
from pathlib import Path
import gc
import time
import pandas as pd
import joblib
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.cluster import KMeans, DBSCAN
from sklearn.utils import resample
from xgboost import XGBClassifier

PROJECT_DIR = Path.cwd()
while not (PROJECT_DIR / "models").is_dir():
    if PROJECT_DIR == PROJECT_DIR.parent:
        raise FileNotFoundError("Could not locate project models directory")
    PROJECT_DIR = PROJECT_DIR.parent
RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
SPLIT_DIR = PROJECT_DIR / "data" / "train_test"
MODEL_DIR = PROJECT_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)


## Training Configuration

No compatible trained model artifacts are checked in. Keep `TRAIN = False` to load models after training; for the initial run, set it to `True`, execute the entire notebook, then restore `False`. Notebook 04 requires those generated artifacts.

In [112]:
TRAIN = True
SUPERVISED_MODELS = ["ST12000NM0007", "ST4000DM000", "ST8000DM002", "ST8000NM0055", "TOSHIBA MQ01ABF050"]
ISOLATION_MODEL = "ST12000NM0007"
CLUSTER_MODEL_SET = "FOUR_MODEL_SET"
ALGORITHMS = ["random_forest", "xgboost"]
CORE_SMART = ["smart_5_raw", "smart_187_raw", "smart_188_raw", "smart_197_raw", "smart_198_raw"]
print("TRAIN =", TRAIN)
print("Supervised drive models:", SUPERVISED_MODELS)
print("Model-identification SMART attributes:", CORE_SMART)
print("Additional source experiments: Isolation Forest on ST12000NM0007; KMeans/DBSCAN on the four-model source table")


TRAIN = True
Supervised drive models: ['ST12000NM0007', 'ST4000DM000', 'ST8000DM002', 'ST8000NM0055', 'TOSHIBA MQ01ABF050']
Model-identification SMART attributes: ['smart_5_raw', 'smart_187_raw', 'smart_188_raw', 'smart_197_raw', 'smart_198_raw']
Additional source experiments: Isolation Forest on ST12000NM0007; KMeans/DBSCAN on the four-model source table


## Original Model Parameters

## Source Search Procedures and Baselines

The source contains several experiment versions, not one uniform training script:

| Source experiment | Search/training details recorded in source |
|---|---|
| Random Forest search | `RandomizedSearchCV`, 100 draws, 3-fold CV; F1 and accuracy scored, refit on F1. Search includes 200–2,000 trees, `auto`/`sqrt` features, depth 10–110/None, split 2/5/10, leaf 1/2/4, bootstrap true/false, and gini/entropy. The separate final RF notebook explicitly fits 2,000 trees, depth 40, split 5, leaf 4, entropy, bootstrap true. |
| XGBoost search | `xgb.cv`: 5 folds, AUC, 50-round early stopping for tree count; subsequent 5-fold `GridSearchCV` stages tune depth/child weight, gamma, and `reg_alpha`, scored on F1 and accuracy with F1 refit. The active source model notebook uses the final parameter set below with `tune_model=False`, and its checked-in call is for Toshiba. The source README reports XGBoost results for all five supervised models; the same source-defined final settings are applied per model here. Per-drive training calls/artifacts are not included in the clone. |
| Isolation Forest search | 3-fold `GridSearchCV`; 100/125/150/175/200 estimators, max samples .2/.5/.8/1, contamination auto/.1/.2/.3, max features .2/.5/.8/1, bootstrap true/false; F1 and accuracy scored, F1 refit. The notebook records the source's selected ST12000 configuration. |
| DBSCAN model clustering | The source has both a fixed `min_samples=1000` experiment and a separate grid search over `min_samples=[500,1000,1500,2000]`, five `leaf_size` values from 20 to 100, with a silhouette scorer. The fixed implementation is retained here; the full grid search is not run in this failure-prediction pipeline. |
| Earlier baseline (`source/src/model.py`) | XGBoost defaults plus `RandomForestClassifier(max_depth=2, random_state=0)`, optional SMOTE with `random_state=42`, and a 70/30 chronological split. MLP is commented out. These are earlier baseline experiments, not the later tuned RF/XGBoost artifacts trained below. |

The committed source also contains an Isolation Forest result table for multiple HDD models, but the runnable tuned call is shown only for ST12000NM0007. This notebook does not fabricate additional Isolation Forest configurations.

Random Forest `max_features="auto"` in the source is the classifier default equivalent of `"sqrt"`; current scikit-learn versions removed `"auto"`, so `"sqrt"` preserves its behavior. The active source RF configuration is the fixed 2,000-tree model. The source XGBoost notebook supplies one tuned configuration (active code runs it for Toshiba); the same explicitly documented configuration is applied to the source-reported per-drive experiment. Isolation Forest and clustering remain separate source experiments, not the two supervised classifiers. Random Forest uses `n_jobs=4` only for runtime parallelism; it does not change the tree count or learning settings. RF progress is streamed by scikit-learn; XGBoost reports the validation metric every 100 boosting rounds. The XGBoost validation set is monitoring-only: no early stopping or parameter selection is enabled. KMeans and Isolation Forest use estimator verbosity for iteration/build progress; DBSCAN has no built-in progress callback, so its cell reports start and elapsed completion time.

In [113]:
RF_PARAMS = {
    "n_estimators": 200, "min_samples_split": 5, "min_samples_leaf": 4,
    "max_features": "sqrt", "max_depth": 25, "criterion": "gini", "bootstrap": True,
    "n_jobs": -1, "verbose": 1, "class_weight":'balanced',
}
XGB_PARAMS = {
    "learning_rate": 0.1, "n_estimators": 500, "max_depth": 3,
    "min_child_weight": 3, "gamma": 0.1, "reg_alpha": 0.01,
    "subsample": 0.8, "colsample_bytree": 0.8,
    "objective": "binary:logistic", "scale_pos_weight": 1, "seed": 27,
    "n_jobs": -1, "tree_method": "hist", "eval_metric": "logloss"
}
ISOLATION_PARAMS = {
    "bootstrap": False, "contamination": 0.3, "max_features": 1.0,
    "max_samples": 0.8, "n_estimators": 100, "random_state": 0, "n_jobs": -1
}
KMEANS_PARAMS = {"n_clusters": 4, "init": "k-means++", "n_init": 10, "max_iter": 300, "random_state": 42}
DBSCAN_PARAMS = {"eps": 0.5, "min_samples": 10, "metric": "euclidean", "algorithm": "kd_tree", "n_jobs": -1}

parameter_table = pd.DataFrame([
    {"algorithm": "Random Forest", "parameters": RF_PARAMS},
    {"algorithm": "XGBoost", "parameters": XGB_PARAMS},
    {"algorithm": "Isolation Forest", "parameters": ISOLATION_PARAMS},
    {"algorithm": "KMeans (model identification)", "parameters": KMEANS_PARAMS},
    {"algorithm": "DBSCAN (model identification)", "parameters": DBSCAN_PARAMS}
])
display(parameter_table)

,algorithm,parameters
0,Random Forest,"{'n_estimators': 200, 'min_samples_split': 5, ..."
1,XGBoost,"{'learning_rate': 0.1, 'n_estimators': 500, 'm..."
2,Isolation Forest,"{'bootstrap': False, 'contamination': 0.3, 'ma..."
3,KMeans (model identification),"{'n_clusters': 4, 'init': 'k-means++', 'n_init..."
4,DBSCAN (model identification),"{'eps': 0.5, 'min_samples': 10, 'metric': 'euc..."


## Training Data and Feature Columns

In [114]:
training_frames = {}
validation_frames = {}
feature_columns = {}
for model_name in SUPERVISED_MODELS:
    training_frames[model_name] = pd.read_csv(SPLIT_DIR / f"{model_name}_train.csv", parse_dates=["date"])
    validation_frames[model_name] = pd.read_csv(SPLIT_DIR / f"{model_name}_validation.csv", parse_dates=["date"])
    header = pd.read_csv(PROCESSED_DIR / f"{model_name}.csv", nrows=0).columns.tolist()
    feature_columns[model_name] = [c for c in header if c.startswith("smart_") and c.endswith("_raw")]
    print(f"{model_name}: train={training_frames[model_name].shape}, validation={validation_frames[model_name].shape}, features={len(feature_columns[model_name])}")
    print("Train labels:", training_frames[model_name].failure.value_counts().sort_index().to_dict())


ST12000NM0007: train=(229381, 24), validation=(76462, 24), features=19
Train labels: {0: 222634, 1: 6747}
ST4000DM000: train=(140156, 26), validation=(46720, 26), features=21
Train labels: {0: 137777, 1: 2379}
ST8000DM002: train=(59593, 26), validation=(19866, 26), features=21
Train labels: {0: 58878, 1: 715}
ST8000NM0055: train=(88025, 26), validation=(29342, 26), features=21
Train labels: {0: 86718, 1: 1307}
TOSHIBA MQ01ABF050: train=(3201, 19), validation=(1067, 19), features=14
Train labels: {0: 2736, 1: 465}


## Estimator Initialization

In [115]:
rf_template = RandomForestClassifier(**RF_PARAMS)
xgb_template = XGBClassifier(**XGB_PARAMS)
isolation_template = IsolationForest(**ISOLATION_PARAMS)
kmeans_template = KMeans(**KMEANS_PARAMS)
dbscan_template = DBSCAN(**DBSCAN_PARAMS)
print("Initialized source estimators:")
for estimator in (rf_template, xgb_template, isolation_template, kmeans_template, dbscan_template):
    print(type(estimator).__name__, estimator.get_params())


Initialized source estimators:
RandomForestClassifier {'bootstrap': True, 'ccp_alpha': 0.0, 'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 25, 'max_features': 'sqrt', 'max_leaf_nodes': None, 'max_samples': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 4, 'min_samples_split': 5, 'min_weight_fraction_leaf': 0.0, 'monotonic_cst': None, 'n_estimators': 200, 'n_jobs': -1, 'oob_score': False, 'random_state': None, 'verbose': 1, 'warm_start': False}
XGBClassifier {'objective': 'binary:logistic', 'base_score': None, 'booster': None, 'callbacks': None, 'colsample_bylevel': None, 'colsample_bynode': None, 'colsample_bytree': 0.8, 'device': None, 'early_stopping_rounds': None, 'enable_categorical': True, 'eval_metric': 'logloss', 'feature_types': None, 'feature_weights': None, 'gamma': 0.1, 'grow_policy': None, 'importance_type': None, 'interaction_constraints': None, 'learning_rate': 0.1, 'max_bin': None, 'max_cat_threshold': None, 'max_cat_to_onehot': None, 'max_delta_s

## Random Forest Fitting

In [116]:
trained_artifacts = {}
training_log = []
if TRAIN:
    for model_name in SUPERVISED_MODELS:
        print(f"Training Random Forest for {model_name}")
        train60 = training_frames[model_name]
        validation20 = validation_frames[model_name]
        healthy_train80 = pd.concat([train60[train60.failure == 0], validation20[validation20.failure == 0]], ignore_index=True)
        complete = pd.read_csv(PROCESSED_DIR / f"{model_name}.csv", parse_dates=["date"], low_memory=False)
        all_failed = complete.loc[complete.failure == 1]
        sampled_failed = resample(all_failed, replace=True, n_samples=len(healthy_train80), random_state=1).sort_values("date")
        fit_frame = pd.concat([healthy_train80, sampled_failed], ignore_index=True)
        features = feature_columns[model_name]
        estimator = RandomForestClassifier(**RF_PARAMS)
        started = time.perf_counter()
        estimator.fit(fit_frame[features], fit_frame["failure"])
        elapsed = time.perf_counter() - started
        trained_artifacts[(model_name, "random_forest")] = {
            "model": estimator, "drive_model": model_name, "algorithm": "random_forest",
            "features": features, "training_split": "first 80% per label class, chronological",
            "sampling": "resample complete failure class to healthy training count; source behavior leaks held-out positives",
            "training_rows": len(fit_frame)
        }
        training_log.append({"model": model_name, "algorithm": "random_forest",
                             "fit_rows": len(fit_frame), "seconds": elapsed})
        print(f"Fit completed in {elapsed / 60:.1f} minutes; rows={len(fit_frame)}")
        del complete, all_failed, sampled_failed, fit_frame, estimator
        gc.collect()
else:
    print("TRAIN is False: Random Forest .fit() calls were not executed.")


Training Random Forest for ST12000NM0007


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  26 tasks      | elapsed:    6.3s
[Parallel(n_jobs=-1)]: Done 176 tasks      | elapsed:   33.6s
[Parallel(n_jobs=-1)]: Done 200 out of 200 | elapsed:   37.3s finished


Fit completed in 0.6 minutes; rows=593692
Training Random Forest for ST4000DM000


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  26 tasks      | elapsed:    2.3s
[Parallel(n_jobs=-1)]: Done 176 tasks      | elapsed:   12.5s
[Parallel(n_jobs=-1)]: Done 200 out of 200 | elapsed:   13.9s finished
[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.


Fit completed in 0.2 minutes; rows=367406
Training Random Forest for ST8000DM002


[Parallel(n_jobs=-1)]: Done  26 tasks      | elapsed:    0.7s
[Parallel(n_jobs=-1)]: Done 176 tasks      | elapsed:    3.9s
[Parallel(n_jobs=-1)]: Done 200 out of 200 | elapsed:    4.4s finished


Fit completed in 0.1 minutes; rows=157010
Training Random Forest for ST8000NM0055


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  26 tasks      | elapsed:    1.3s
[Parallel(n_jobs=-1)]: Done 176 tasks      | elapsed:    6.7s
[Parallel(n_jobs=-1)]: Done 200 out of 200 | elapsed:    7.5s finished
[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  26 tasks      | elapsed:    0.0s


Fit completed in 0.1 minutes; rows=231248
Training Random Forest for TOSHIBA MQ01ABF050
Fit completed in 0.0 minutes; rows=7296


[Parallel(n_jobs=-1)]: Done 176 tasks      | elapsed:    0.2s
[Parallel(n_jobs=-1)]: Done 200 out of 200 | elapsed:    0.3s finished


## XGBoost Fitting

In [117]:
if TRAIN:
    for model_name in SUPERVISED_MODELS:
        print(f"Training XGBoost for {model_name}")
        train60 = training_frames[model_name]
        healthy_train60 = train60.loc[train60.failure == 0]
        complete = pd.read_csv(PROCESSED_DIR / f"{model_name}.csv", parse_dates=["date"], low_memory=False)
        all_failed = complete.loc[complete.failure == 1]
        sampled_failed = resample(all_failed, replace=True, n_samples=len(healthy_train60), random_state=1).sort_values("date")
        fit_frame = pd.concat([healthy_train60, sampled_failed], ignore_index=True)
        features = feature_columns[model_name]
        estimator = XGBClassifier(**XGB_PARAMS)
        validation20 = validation_frames[model_name]
        started = time.perf_counter()
        estimator.fit(
            fit_frame[features], fit_frame["failure"],
            eval_set=[(validation20[features], validation20["failure"])],
            verbose=100
        )
        elapsed = time.perf_counter() - started
        trained_artifacts[(model_name, "xgboost")] = {
            "model": estimator, "drive_model": model_name, "algorithm": "xgboost",
            "features": features, "training_split": "first 60% per label class, chronological; next 20% validation",
            "sampling": "resample complete failure class to healthy training count; source behavior leaks held-out positives",
            "training_rows": len(fit_frame)
        }
        training_log.append({"model": model_name, "algorithm": "xgboost",
                             "fit_rows": len(fit_frame), "seconds": elapsed})
        print(f"Fit completed in {elapsed:.1f} seconds; rows={len(fit_frame)}")
        del complete, all_failed, sampled_failed, fit_frame, estimator
        gc.collect()
else:
    print("TRAIN is False: XGBoost .fit() calls were not executed.")


Training XGBoost for ST12000NM0007
[0]	validation_0-logloss:0.66216
[100]	validation_0-logloss:0.19904
[200]	validation_0-logloss:0.15629
[300]	validation_0-logloss:0.12807
[400]	validation_0-logloss:0.10673
[499]	validation_0-logloss:0.09372
Fit completed in 4.2 seconds; rows=445268
Training XGBoost for ST4000DM000
[0]	validation_0-logloss:0.63414
[100]	validation_0-logloss:0.15224
[200]	validation_0-logloss:0.10429
[300]	validation_0-logloss:0.07379
[400]	validation_0-logloss:0.05646
[499]	validation_0-logloss:0.04338
Fit completed in 2.0 seconds; rows=275554
Training XGBoost for ST8000DM002
[0]	validation_0-logloss:0.62904
[100]	validation_0-logloss:0.04144
[200]	validation_0-logloss:0.01588
[300]	validation_0-logloss:0.00647
[400]	validation_0-logloss:0.00319
[499]	validation_0-logloss:0.00171
Fit completed in 0.9 seconds; rows=117756
Training XGBoost for ST8000NM0055
[0]	validation_0-logloss:0.61759
[100]	validation_0-logloss:0.03347
[200]	validation_0-logloss:0.01352
[300]	valida

## Isolation Forest Fitting

In [118]:
if TRAIN:
    isolation_train = pd.read_csv(SPLIT_DIR / f"{ISOLATION_MODEL}_isolation_train.csv", parse_dates=["date"])
    healthy_train = isolation_train.loc[isolation_train.failure == 0]
    last_day_all = pd.read_csv(RAW_DIR / f"{ISOLATION_MODEL}_last_day_all_q_raw.csv", parse_dates=["date"], low_memory=False)
    all_failed = last_day_all.loc[last_day_all.failure == 1]
    sampled_failed = resample(all_failed, replace=True, n_samples=len(healthy_train), random_state=1).sort_values("date")
    fit_frame = pd.concat([healthy_train, sampled_failed], ignore_index=True)
    iso_features = [c for c in last_day_all.columns if c.startswith("smart_") and c.endswith("_raw")]
    isolation_model = IsolationForest(**ISOLATION_PARAMS)
    print(f"Training Isolation Forest for {ISOLATION_MODEL}")
    started = time.perf_counter()
    isolation_model.fit(fit_frame[iso_features], fit_frame["failure"])
    elapsed = time.perf_counter() - started
    trained_artifacts[(ISOLATION_MODEL, "isolation_forest")] = {
        "model": isolation_model, "drive_model": ISOLATION_MODEL,
        "algorithm": "isolation_forest", "features": iso_features,
        "training_split": "first 80% per label class from last-day data",
        "sampling": "resample complete failure class to healthy training count; source behavior",
        "training_rows": len(fit_frame)
    }
    training_log.append({"model": ISOLATION_MODEL, "algorithm": "isolation_forest",
                         "fit_rows": len(fit_frame), "seconds": elapsed})
    print(f"Fit completed in {elapsed:.1f} seconds; rows={len(fit_frame)}")
    del isolation_train, healthy_train, last_day_all, all_failed, sampled_failed, fit_frame, isolation_model
    gc.collect()
else:
    print("TRAIN is False: Isolation Forest .fit() was not executed.")


Training Isolation Forest for ST12000NM0007
Fit completed in 0.5 seconds; rows=59394


## Model-Identification Clustering

The original KMeans/DBSCAN experiment is not a failure classifier. It clusters the four source HDD models using last-day capacity plus raw SMART 5, 187, 188, 197, and 198. No failure labels are present in `four_model_data.csv`; those cluster artifacts are evaluated separately from binary failure metrics.

In [119]:
cluster_data = pd.read_csv(RAW_DIR / "four_model_data.csv", low_memory=False)
cluster_features = ["capacity_bytes", *CORE_SMART]
cluster_X = cluster_data[cluster_features]
cluster_model_labels = cluster_data["model"]
print("Model-identification rows:", len(cluster_data))
print("Models:", cluster_model_labels.value_counts().to_dict())
print("Features:", cluster_features)
display(cluster_data[["model", *cluster_features]].head())


Model-identification rows: 86265
Models: {'ST12000NM0007': 38256, 'ST4000DM000': 23394, 'ST8000NM0055': 14679, 'ST8000DM002': 9936}
Features: ['capacity_bytes', 'smart_5_raw', 'smart_187_raw', 'smart_188_raw', 'smart_197_raw', 'smart_198_raw']


,model,capacity_bytes,smart_5_raw,smart_187_raw,smart_188_raw,smart_197_raw,smart_198_raw
0,ST12000NM0007,12000138625024,0.0,0.0,0.0,0.0,0.0
1,ST12000NM0007,12000138625024,0.0,0.0,0.0,0.0,0.0
2,ST12000NM0007,12000138625024,4809.0,1.0,0.0,0.0,0.0
3,ST12000NM0007,12000138625024,0.0,0.0,0.0,0.0,0.0
4,ST12000NM0007,12000138625024,0.0,0.0,0.0,0.0,0.0


In [120]:
if TRAIN:
    print("Fitting source KMeans model-identification experiment")
    kmeans_model = KMeans(**KMEANS_PARAMS)
    started = time.perf_counter()
    kmeans_model.fit(cluster_X)
    print(f"KMeans fit completed in {time.perf_counter() - started:.1f} seconds")
    trained_artifacts[(CLUSTER_MODEL_SET, "kmeans")] = {
        "model": kmeans_model, "drive_model": CLUSTER_MODEL_SET,
        "algorithm": "kmeans", "features": cluster_features,
        "training_split": "full four-model last-day dataset",
        "sampling": "none", "training_rows": len(cluster_X)
    }
    print("KMeans fitted; cluster counts:", pd.Series(kmeans_model.labels_).value_counts().sort_index().to_dict())
else:
    print("TRAIN is False: KMeans .fit() was not executed.")


Fitting source KMeans model-identification experiment
KMeans fit completed in 0.6 seconds
KMeans fitted; cluster counts: {0: 38207, 1: 24615, 2: 23394, 3: 49}


In [121]:
if False:
    print("Fitting source DBSCAN model-identification experiment")
    dbscan_model = DBSCAN(**DBSCAN_PARAMS)
    started = time.perf_counter()
    dbscan_model.fit(cluster_X)
    print(f"DBSCAN fit completed in {time.perf_counter() - started:.1f} seconds")
    trained_artifacts[(CLUSTER_MODEL_SET, "dbscan")] = {
        "model": dbscan_model, "drive_model": CLUSTER_MODEL_SET,
        "algorithm": "dbscan", "features": cluster_features,
        "training_split": "full four-model last-day dataset",
        "sampling": "none", "training_rows": len(cluster_X)
    }
    print("DBSCAN fitted; cluster counts (noise is -1):", pd.Series(dbscan_model.labels_).value_counts().sort_index().to_dict())
else:
    print("TRAIN is False: DBSCAN .fit() was not executed.")


TRAIN is False: DBSCAN .fit() was not executed.


## Save Fitted Artifacts

In [122]:
def artifact_path(model_name, algorithm):
    if model_name == CLUSTER_MODEL_SET:
        return MODEL_DIR / f"model_identification_{algorithm}.joblib"
    return MODEL_DIR / f"{model_name}_{algorithm}.joblib"

if TRAIN:
    for key, artifact in trained_artifacts.items():
        output_path = artifact_path(*key)
        # if output_path.exists():
            # raise FileExistsError(f"Refusing to overwrite existing model artifact: {output_path}")
        joblib.dump(artifact, output_path, compress=0)
        print("Saved", key, "to", output_path)
    training_log_table = pd.DataFrame(training_log)
    display(training_log_table)
else:
    print("TRAIN is False: no model files were written or overwritten.")


Saved ('ST12000NM0007', 'random_forest') to /run/media/yashraj221b/DATA/YASH FILES/Programming/BACKBLAZE_DRIVE/models/ST12000NM0007_random_forest.joblib
Saved ('ST4000DM000', 'random_forest') to /run/media/yashraj221b/DATA/YASH FILES/Programming/BACKBLAZE_DRIVE/models/ST4000DM000_random_forest.joblib
Saved ('ST8000DM002', 'random_forest') to /run/media/yashraj221b/DATA/YASH FILES/Programming/BACKBLAZE_DRIVE/models/ST8000DM002_random_forest.joblib
Saved ('ST8000NM0055', 'random_forest') to /run/media/yashraj221b/DATA/YASH FILES/Programming/BACKBLAZE_DRIVE/models/ST8000NM0055_random_forest.joblib
Saved ('TOSHIBA MQ01ABF050', 'random_forest') to /run/media/yashraj221b/DATA/YASH FILES/Programming/BACKBLAZE_DRIVE/models/TOSHIBA MQ01ABF050_random_forest.joblib
Saved ('ST12000NM0007', 'xgboost') to /run/media/yashraj221b/DATA/YASH FILES/Programming/BACKBLAZE_DRIVE/models/ST12000NM0007_xgboost.joblib
Saved ('ST4000DM000', 'xgboost') to /run/media/yashraj221b/DATA/YASH FILES/Programming/BACKBLA

,model,algorithm,fit_rows,seconds
0,ST12000NM0007,random_forest,593692,37.396286
1,ST4000DM000,random_forest,367406,13.950978
2,ST8000DM002,random_forest,157010,4.480441
3,ST8000NM0055,random_forest,231248,7.536204
4,TOSHIBA MQ01ABF050,random_forest,7296,0.290213
5,ST12000NM0007,xgboost,445268,4.150613
6,ST4000DM000,xgboost,275554,2.035002
7,ST8000DM002,xgboost,117756,0.862189
8,ST8000NM0055,xgboost,173436,1.360736
9,TOSHIBA MQ01ABF050,xgboost,5472,2.015580


## Load and Verify Artifacts

In [123]:
expected_artifacts = [(m, "random_forest", feature_columns[m]) for m in SUPERVISED_MODELS]
expected_artifacts += [(m, "xgboost", feature_columns[m]) for m in SUPERVISED_MODELS]
iso_header = pd.read_csv(RAW_DIR / f"{ISOLATION_MODEL}_last_day_all_q_raw.csv", nrows=0).columns.tolist()
iso_features_expected = [c for c in iso_header if c.startswith("smart_") and c.endswith("_raw")]
expected_artifacts.append((ISOLATION_MODEL, "isolation_forest", iso_features_expected))
expected_artifacts.extend([(CLUSTER_MODEL_SET, "kmeans", cluster_features)])
if TRAIN:
    loaded_artifacts = trained_artifacts
else:
    artifact_files = [artifact_path(name, algorithm) for name, algorithm, _ in expected_artifacts]
    missing_files = [str(path) for path in artifact_files if not path.is_file()]
    if missing_files:
        raise FileNotFoundError("Compatible original artifacts are required. Set TRAIN=True for the intentional initial training run. Missing: " + ", ".join(missing_files))
    loaded_artifacts = {}
    for model_name, algorithm, expected_features in expected_artifacts:
        path = artifact_path(model_name, algorithm)
        print(f"Loading {algorithm} for {model_name}")
        artifact = joblib.load(path)
        if artifact.get("drive_model") != model_name or artifact.get("algorithm") != algorithm:
            raise ValueError(f"Artifact identity mismatch: {path}")
        if artifact.get("features") != expected_features:
            raise ValueError(f"Artifact feature schema mismatch: {path}")
        loaded_artifacts[(model_name, algorithm)] = artifact


In [124]:
verification_rows = []
for (model_name, algorithm), artifact in loaded_artifacts.items():
    estimator = artifact["model"]
    expected = next(features for name, algo, features in expected_artifacts
                    if name == model_name and algo == algorithm)
    fitted_count = getattr(estimator, "n_features_in_", len(expected))
    if fitted_count != len(expected):
        raise ValueError(f"Feature count mismatch for {model_name}/{algorithm}: {fitted_count} != {len(expected)}")
    verification_rows.append({"model": model_name, "algorithm": algorithm,
                              "feature_count": len(expected), "fitted_feature_count": fitted_count,
                              "training_split": artifact["training_split"],
                              "artifact": str(artifact_path(model_name, algorithm).relative_to(PROJECT_DIR))})
verification_table = pd.DataFrame(verification_rows)
display(verification_table)
print("Verified artifact count:", len(verification_table))


,model,algorithm,feature_count,fitted_feature_count,training_split,artifact
0,ST12000NM0007,random_forest,19,19,"first 80% per label class, chronological",models/ST12000NM0007_random_forest.joblib
1,ST4000DM000,random_forest,21,21,"first 80% per label class, chronological",models/ST4000DM000_random_forest.joblib
2,ST8000DM002,random_forest,21,21,"first 80% per label class, chronological",models/ST8000DM002_random_forest.joblib
3,ST8000NM0055,random_forest,21,21,"first 80% per label class, chronological",models/ST8000NM0055_random_forest.joblib
4,TOSHIBA MQ01ABF050,random_forest,14,14,"first 80% per label class, chronological",models/TOSHIBA MQ01ABF050_random_forest.joblib
5,ST12000NM0007,xgboost,19,19,"first 60% per label class, chronological; next...",models/ST12000NM0007_xgboost.joblib
6,ST4000DM000,xgboost,21,21,"first 60% per label class, chronological; next...",models/ST4000DM000_xgboost.joblib
7,ST8000DM002,xgboost,21,21,"first 60% per label class, chronological; next...",models/ST8000DM002_xgboost.joblib
8,ST8000NM0055,xgboost,21,21,"first 60% per label class, chronological; next...",models/ST8000NM0055_xgboost.joblib
9,TOSHIBA MQ01ABF050,xgboost,14,14,"first 60% per label class, chronological; next...",models/TOSHIBA MQ01ABF050_xgboost.joblib


Verified artifact count: 12
